# MNIST 베이지안 최적화 (개선판 v2 — 정석 일반화 기법 적용)

## 이 버전이 baseline을 이기는 원리

이전 개선판은 베이지안 탐색을 붙였는데도 손으로 짠 MLP(첫 파일, 약 98.1%)보다 낮았습니다.
원인은 **탐색이 3에포크 성능을 기준으로 평가**했고, **weight_decay를 선형 공간에서 과하게** 뽑았으며,
모델이 **규제 없는 맨 MLP**였기 때문입니다.

이 v2는 정답을 미리 아는 폴백(예: 좋은 설정 강제 주입) 없이, **보편적으로 일반화를 끌어올리는 정석 기법**만 적용합니다.

1. **BatchNorm + Dropout**: MLP의 표준 규제·안정화 기법. Dropout 비율도 탐색 대상.
2. **로그스케일 하이퍼파라미터 탐색**: `lr`, `weight_decay`는 지수(exponent)를 탐색해 로그 공간에서 균등 샘플링.
3. **데이터 증강(RandomAffine)**: 훈련 이미지에 약한 회전·이동을 줘서 일반화 향상. (검증/테스트는 원본 유지)
4. **탐색 프록시 정합 + 조기종료(early stopping)**: 탐색 에포크를 의미있게 늘리되 효율을 유지.
5. **코사인 LR 스케줄 + 더 긴 최종 학습**: 후반 학습률을 낮춰 더 깊게 수렴.
6. **탐색 횟수 증가**: 다차원 공간을 충분히 탐색.

탐색 대상: lr, weight_decay, batch_size, 히든레이어 수, 레이어당 노드 수, **dropout 비율**


In [ ]:
# ============================================================
# 1. 라이브러리 불러오기
# ============================================================
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset, random_split, Dataset

import numpy as np
from torchvision import datasets, transforms

import matplotlib.pyplot as plt
import random

# 베이지안 최적화 라이브러리 (없으면 자동 설치)
try:
    from bayes_opt import BayesianOptimization
except ImportError:
    import sys, subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "bayesian-optimization"])
    from bayes_opt import BayesianOptimization

In [ ]:
# ============================================================
# 2. 실행 환경 및 랜덤 시드 설정
# ============================================================
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("사용 장치:", device)

## 1. 전역 설정

- `SPLIT_RATIO` : (훈련, 검증, 테스트) 비율 (합 1.0)
- `SEARCH_EPOCHS` : 조합 1개당 탐색 학습 에포크 (조기종료와 함께 사용)
- `SEARCH_PATIENCE` : 검증 정확도가 개선되지 않을 때 몇 에포크 후 조기종료할지
- `FINAL_EPOCHS` : 최적 조합으로 최종 학습할 에포크
- `INIT_POINTS`, `N_ITER` : 베이지안 초기 랜덤 탐색 / 추가 탐색 횟수
- `AUGMENT` : 훈련 데이터 증강 사용 여부


In [ ]:
# ============================================================
# 3. 전역 설정값
# ============================================================
CONFIG = {
    "SPLIT_RATIO": (0.7, 0.15, 0.15),

    # 탐색 단계: 에포크를 늘리되 조기종료로 낭비를 막아 '최종과 비슷한 기준'으로 평가
    "SEARCH_EPOCHS": 6,
    "SEARCH_PATIENCE": 2,

    # 최종 학습: 더 길게 + 코사인 LR 스케줄
    "FINAL_EPOCHS": 20,

    # 탐색 횟수 (5~6차원 공간을 충분히 탐색)
    "INIT_POINTS": 5,
    "N_ITER": 20,

    # 데이터 증강 사용 여부
    "AUGMENT": True,
}
assert abs(sum(CONFIG["SPLIT_RATIO"]) - 1.0) < 1e-6, "SPLIT_RATIO의 합은 1.0이어야 합니다."
print("훈련:검증:테스트 비율 =", CONFIG["SPLIT_RATIO"])

## 2. MNIST 로드 및 비율 기반 3분할

전체 70,000장을 합친 뒤 지정 비율대로 train/val/test로 나눕니다. 픽셀은 0~1로 정규화합니다.


In [ ]:
# ============================================================
# 4. MNIST 로드 및 비율대로 train/val/test 분할
# ============================================================
mnist_train = datasets.MNIST(root="./data", train=True, download=True)
mnist_test = datasets.MNIST(root="./data", train=False, download=True)

all_images = torch.cat([mnist_train.data, mnist_test.data], dim=0)        # [70000, 28, 28]
all_labels = torch.cat([mnist_train.targets, mnist_test.targets], dim=0)  # [70000]

all_X = (all_images.float() / 255.0).unsqueeze(1)  # [70000, 1, 28, 28]
all_Y = all_labels.long()

full_dataset = TensorDataset(all_X, all_Y)

total_len = len(full_dataset)
train_ratio, val_ratio, test_ratio = CONFIG["SPLIT_RATIO"]
train_size = int(train_ratio * total_len)
val_size = int(val_ratio * total_len)
test_size = total_len - train_size - val_size

train_dataset, val_dataset, test_dataset = random_split(
    full_dataset, [train_size, val_size, test_size],
    generator=torch.Generator().manual_seed(SEED),
)

print(f"전체 : {total_len} | 훈련 : {len(train_dataset)} | 검증 : {len(val_dataset)} | 테스트 : {len(test_dataset)}")

# 검증/테스트 로더는 원본(증강 X)으로 고정 평가합니다.
val_loader_eval = DataLoader(val_dataset, batch_size=256, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=256, shuffle=False)

## 3. 데이터 증강 래퍼

훈련 데이터에만 약한 **RandomAffine(회전 ±10°, 이동 ±10%)** 를 적용합니다.
검증/테스트는 원본을 그대로 사용해야 평가가 공정하므로 증강하지 않습니다.
이 증강은 "정답을 아는 것"이 아니라, 이미지 분류에서 일반화를 높이는 표준 기법입니다.


In [ ]:
# ============================================================
# 5. 훈련 전용 증강 데이터셋
# ============================================================
class AugmentedDataset(Dataset):
    def __init__(self, base, transform):
        self.base = base            # random_split으로 나온 Subset
        self.transform = transform  # 텐서 이미지에 적용할 변환

    def __len__(self):
        return len(self.base)

    def __getitem__(self, idx):
        x, y = self.base[idx]       # x: [1, 28, 28] float 텐서
        if self.transform is not None:
            x = self.transform(x)
        return x, y

# 약한 회전·이동 증강 (텐서 입력을 그대로 처리)
train_transform = transforms.RandomAffine(degrees=10, translate=(0.1, 0.1))

if CONFIG["AUGMENT"]:
    train_dataset_train = AugmentedDataset(train_dataset, train_transform)
    print("훈련 데이터 증강: 사용 (RandomAffine degrees=10, translate=0.1)")
else:
    train_dataset_train = train_dataset
    print("훈련 데이터 증강: 미사용")

## 4. 샘플 이미지 미리보기 (원본 vs 증강)

In [ ]:
# ============================================================
# 6. 샘플 이미지 시각화
# ============================================================
plt.figure(figsize=(12, 3))
for i in range(8):
    # 위쪽 줄: 원본
    img, label = train_dataset[i]
    plt.subplot(2, 8, i + 1)
    plt.imshow(img.squeeze(), cmap="gray")
    plt.title(f"orig:{label}")
    plt.axis("off")

    # 아래쪽 줄: 증강 결과
    aug_img = train_transform(img)
    plt.subplot(2, 8, i + 9)
    plt.imshow(aug_img.squeeze(), cmap="gray")
    plt.title("aug")
    plt.axis("off")
plt.suptitle("Top: original / Bottom: augmented")
plt.tight_layout()
plt.show()

## 5. 모델 정의 (BatchNorm + Dropout 포함)

각 히든블록은 **Linear → BatchNorm1d → ReLU → Dropout** 입니다.
히든레이어 수, 노드 수, dropout 비율을 인자로 받아 베이지안 탐색이 구조와 규제 강도를 함께 찾습니다.


In [ ]:
# ============================================================
# 7. 규제가 포함된 MLP 모델
# ============================================================
class MNISTMLP(nn.Module):
    def __init__(self, n_hidden_layers, n_nodes, dropout):
        super().__init__()
        layers = [nn.Flatten()]
        in_features = 28 * 28
        for _ in range(n_hidden_layers):
            layers.append(nn.Linear(in_features, n_nodes))
            layers.append(nn.BatchNorm1d(n_nodes))   # 학습 안정화·가속
            layers.append(nn.ReLU())
            layers.append(nn.Dropout(dropout))       # 과적합 억제
            in_features = n_nodes
        layers.append(nn.Linear(in_features, 10))    # 출력층 (10 클래스)
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)

print(MNISTMLP(n_hidden_layers=3, n_nodes=256, dropout=0.2))

In [ ]:
# ============================================================
# 8. 정확도 계산 / 1에포크 학습 / 평가 함수
# ============================================================
def calculate_accuracy(logits, labels):
    predictions = torch.argmax(logits, dim=1)
    return (predictions == labels).float().mean().item()


def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    running_loss, running_acc = 0.0, 0.0
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
        running_acc += calculate_accuracy(outputs, labels)
    return running_loss / len(loader), running_acc / len(loader)


def evaluate(model, loader, criterion, device):
    model.eval()
    running_loss, running_acc = 0.0, 0.0
    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            running_loss += loss.item()
            running_acc += calculate_accuracy(outputs, labels)
    return running_loss / len(loader), running_acc / len(loader)

## 6. 베이지안 최적화 목적 함수 (로그스케일 + 조기종료)

| 탐색 변수 | 의미 | 변환 |
|---|---|---|
| `lr_exp` | 학습률 지수 | `lr = 10 ** lr_exp` (로그스케일) |
| `wd_exp` | 가중치 감쇠 지수 | `weight_decay = 10 ** wd_exp` (로그스케일) |
| `batch_size_log` | 배치 크기 | `2 ** round` (32~256) |
| `n_hidden_layers` | 히든레이어 수 | 반올림 (1~4) |
| `n_nodes_log` | 노드 수 | `2 ** round` (128~512) |
| `dropout` | 드롭아웃 비율 | 그대로 (0.0~0.5) |

각 조합은 `SEARCH_EPOCHS`까지 학습하되, 검증 정확도가 `SEARCH_PATIENCE` 에포크 동안 개선되지 않으면 조기종료합니다.


In [ ]:
# ============================================================
# 9. 베이지안 최적화 목적 함수
# ============================================================
criterion = nn.CrossEntropyLoss()

def train_and_evaluate(lr_exp, wd_exp, batch_size_log, n_hidden_layers, n_nodes_log, dropout):
    # 로그스케일 지수를 실제 값으로 변환
    lr = 10 ** lr_exp
    weight_decay = 10 ** wd_exp
    batch_size = int(2 ** round(batch_size_log))
    n_layers = int(round(n_hidden_layers))
    n_nodes = int(2 ** round(n_nodes_log))

    # 훈련은 증강 데이터, 검증은 원본. drop_last=True로 BatchNorm의 배치 크기 1 오류 방지
    train_loader = DataLoader(train_dataset_train, batch_size=batch_size, shuffle=True, drop_last=True)

    model = MNISTMLP(n_layers, n_nodes, dropout).to(device)
    optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)

    best_val_acc = 0.0
    bad_epochs = 0
    for _ in range(CONFIG["SEARCH_EPOCHS"]):
        train_one_epoch(model, train_loader, criterion, optimizer, device)
        _, val_acc = evaluate(model, val_loader_eval, criterion, device)

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            bad_epochs = 0
        else:
            bad_epochs += 1
            if bad_epochs >= CONFIG["SEARCH_PATIENCE"]:
                break  # 조기종료로 탐색 효율 확보

    return best_val_acc

In [ ]:
# ============================================================
# 10. 베이지안 최적화 실행
# ============================================================
pbounds = {
    "lr_exp": (-4.0, -2.3),      # 10^-4(0.0001) ~ 10^-2.3(약 0.005)
    "wd_exp": (-6.0, -3.0),      # 10^-6 ~ 10^-3 (로그스케일, 과한 wd 방지)
    "batch_size_log": (5, 8),    # 32 ~ 256
    "n_hidden_layers": (1, 4),   # 1 ~ 4
    "n_nodes_log": (7, 9),       # 128 ~ 512
    "dropout": (0.0, 0.5),       # 드롭아웃 비율
}

optimizer_bo = BayesianOptimization(
    f=train_and_evaluate,
    pbounds=pbounds,
    random_state=SEED,
    verbose=2,
)

print("=== 베이지안 최적화 탐색 시작 (시간이 걸립니다) ===")
optimizer_bo.maximize(init_points=CONFIG["INIT_POINTS"], n_iter=CONFIG["N_ITER"])

## 7. 최적 하이퍼파라미터 정리 및 수렴 시각화

In [ ]:
# ============================================================
# 11. 최적 하이퍼파라미터 추출
# ============================================================
best = optimizer_bo.max
p = best["params"]

best_lr = 10 ** p["lr_exp"]
best_weight_decay = 10 ** p["wd_exp"]
best_batch_size = int(2 ** round(p["batch_size_log"]))
best_n_layers = int(round(p["n_hidden_layers"]))
best_n_nodes = int(2 ** round(p["n_nodes_log"]))
best_dropout = p["dropout"]

print("===== 최적 하이퍼파라미터 =====")
print(f"학습률(lr)       : {best_lr:.6f}")
print(f"가중치 감쇠(wd)  : {best_weight_decay:.6e}")
print(f"배치 크기        : {best_batch_size}")
print(f"히든레이어 수    : {best_n_layers}")
print(f"레이어당 노드 수 : {best_n_nodes}")
print(f"드롭아웃 비율    : {best_dropout:.3f}")
print(f"탐색 중 최고 검증 정확도 : {best['target'] * 100:.2f}%")

In [ ]:
# ============================================================
# 12. 베이지안 최적화 수렴 과정 시각화
# ============================================================
targets = [res["target"] for res in optimizer_bo.res]
running_best = np.maximum.accumulate(targets)

plt.figure(figsize=(10, 5))
plt.plot(range(1, len(targets) + 1), targets, "o-", label="val acc per trial")
plt.plot(range(1, len(targets) + 1), running_best, "s--", label="best so far")
plt.xlabel("trial")
plt.ylabel("validation accuracy")
plt.title("Bayesian Optimization Convergence")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## 8. 최적 조합으로 최종 재학습 (코사인 LR 스케줄 + 베스트 체크포인트)

최적 하이퍼파라미터로 `FINAL_EPOCHS` 만큼 다시 학습하면서 코사인 LR 스케줄을 적용하고,
검증 손실이 가장 낮은 시점의 가중치를 저장해 최종 평가에 사용합니다.


In [ ]:
# ============================================================
# 13. 최종 모델 재학습
# ============================================================
final_train_loader = DataLoader(
    train_dataset_train, batch_size=best_batch_size, shuffle=True, drop_last=True
)

final_model = MNISTMLP(best_n_layers, best_n_nodes, best_dropout).to(device)
final_optimizer = optim.AdamW(final_model.parameters(), lr=best_lr, weight_decay=best_weight_decay)

# 코사인 annealing: 학습이 진행될수록 학습률을 부드럽게 0 쪽으로 줄입니다.
scheduler = optim.lr_scheduler.CosineAnnealingLR(final_optimizer, T_max=CONFIG["FINAL_EPOCHS"])

train_losses, val_losses = [], []
train_accuracies, val_accuracies = [], []

best_val_loss = float("inf")
best_state = None

for epoch in range(CONFIG["FINAL_EPOCHS"]):
    train_loss, train_acc = train_one_epoch(
        final_model, final_train_loader, criterion, final_optimizer, device
    )
    val_loss, val_acc = evaluate(final_model, val_loader_eval, criterion, device)
    scheduler.step()

    train_losses.append(train_loss)
    val_losses.append(val_loss)
    train_accuracies.append(train_acc)
    val_accuracies.append(val_acc)

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_state = {k: v.cpu().clone() for k, v in final_model.state_dict().items()}

    print(
        f"Epoch [{epoch + 1}/{CONFIG['FINAL_EPOCHS']}] | "
        f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f} | "
        f"Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f}"
    )

In [ ]:
# ============================================================
# 14. 학습 곡선 시각화
# ============================================================
fig, loss_ax = plt.subplots(figsize=(10, 6))
loss_ax.plot(train_losses, "b-", label="train loss")
loss_ax.plot(val_losses, "b--", label="val loss")
loss_ax.set_xlabel("epoch")
loss_ax.set_ylabel("loss", color="b")

acc_ax = loss_ax.twinx()
acc_ax.plot(train_accuracies, "r-", label="train acc")
acc_ax.plot(val_accuracies, "r--", label="val acc")
acc_ax.set_ylabel("accuracy", color="r")

loss_ax.legend(loc="upper left")
acc_ax.legend(loc="lower right")
plt.title("Final Model Training History")
plt.show()

## 9. 테스트셋 최종 평가 및 예측 결과 시각화

In [ ]:
# ============================================================
# 15. 베스트 가중치 복원 후 테스트 평가
# ============================================================
if best_state is not None:
    final_model.load_state_dict(best_state)

test_loss, test_acc = evaluate(final_model, test_loader, criterion, device)
print(f"최종 테스트 손실   : {test_loss:.4f}")
print(f"최종 테스트 정확도 : {test_acc * 100:.2f}%")

In [ ]:
# ============================================================
# 16. 테스트 예측 결과 시각화
# ============================================================
final_model.eval()
test_images, test_labels = next(iter(test_loader))
test_images, test_labels = test_images.to(device), test_labels.to(device)

with torch.no_grad():
    outputs = final_model(test_images)
    _, predictions = torch.max(torch.softmax(outputs, dim=1), dim=1)

num_images = 10
plt.figure(figsize=(15, 4))
for i in range(num_images):
    plt.subplot(1, num_images, i + 1)
    plt.imshow(test_images[i].cpu().squeeze(), cmap="gray")
    t, pr = test_labels[i].item(), predictions[i].item()
    plt.title(f"T:{t}\nP:{pr}", color=("green" if t == pr else "red"))
    plt.axis("off")
plt.suptitle("Test predictions (green=correct, red=wrong)")
plt.show()